# SENTRIX — UCF-Crime-DVS Event-Based Anomaly Training

**Notebook:** `G:\Sentrix\training\SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb`
**Dataset :** `G:\event_frame_duration533326\event_frame_duration533326`  *(stays where it is — never copied into `G:\Capstone\data`)*

## Why this is a separate notebook

UCF-Crime-DVS is **event (DVS) data stored as `.npz`**, not JPG/PNG frames. It is a
**weakly-supervised video anomaly detection benchmark** whose reference implementation uses
spiking neural networks (SpikingJelly / Spikingformer / multi-scale spiking fusion).
Feeding these files into the old image-classifier pipeline would be wrong on three counts:
the input is a spatio-temporal event tensor, the supervision is video-level (not frame-level),
and the useful output is a continuous **anomaly score**, not a 14-way label.

## What this notebook produces

A SENTRIX-facing record per clip:

```text
video_id
clip_id
anomaly_score        <- the value the TCI fusion layer should consume
predicted_category   <- auxiliary, weakly supervised - context only
confidence
```

## The rule that governs this notebook

> **The 14-class head is auxiliary. Only `anomaly_score` is wired into TCI.**

The category head is trained on *video-level* (weak) labels propagated to the top-scoring
segments. It is useful for operator context — "this looks like Shoplifting" — but it is not
frame-level ground truth and must not be treated as one.

## Real-data policy (identical to the V2/V3 notebooks)

- No synthetic event streams, no synthetic frames, no invented labels
- The train/test partition comes from the **official** `train_split.txt` / `test_split.txt`
- Frame-level evaluation uses the official `gt-ucf.npy` / `gt-ucf-dic.pickle` when present;
  otherwise the notebook reports video-level AUC and says so, rather than fabricating GT
- Nothing under the dataset root is ever written to

## Section map

| # | Section |
|---|---|
| 1 | Environment / GPU verification |
| 2 | UCF-Crime-DVS path configuration |
| 3 | NPZ inventory |
| 4 | Inspect NPZ structure |
| 5 | Map the 14 UCF-Crime-DVS categories |
| 6 | Load the official `train_split.txt` / `test_split.txt` |
| 7 | Build train / validation / test samples |
| 8 | Load NPZ event data |
| 9 | Event preprocessing |
| 10 | Event representation / tensor conversion (+ cache) |
| 11 | UCF-Crime-DVS model (SNN encoder + multi-scale fusion) |
| 12 | Training |
| 13 | Validation |
| 14 | Test evaluation |
| 15 | Confusion matrix / classification metrics |
| 16 | Save model |
| 17 | Save preprocessing + class metadata |
| 18 | Save results + SENTRIX inference API |

---
# CELL 1 — Environment / GPU verification

In [14]:
import sys, os, json, math, time, random, shutil, pickle, platform, warnings, hashlib
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def _ver(name):
    try:
        import importlib
        m = importlib.import_module(name)
        return getattr(m, "__version__", "installed")
    except Exception as e:
        return f"NOT INSTALLED ({type(e).__name__})"

print("=" * 64)
print("ENVIRONMENT")
print("=" * 64)
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("PyTorch     :", torch.__version__)
print("NumPy       :", np.__version__)
print("SpikingJelly:", _ver("spikingjelly"))
print("scikit-learn:", _ver("sklearn"))
print("pandas      :", _ver("pandas"))
print("timm        :", _ver("timm"))

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("\nDEVICE      :", DEVICE)
if DEVICE == "cuda":
    print("CUDA runtime:", torch.version.cuda)
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.1f} GB | SM {p.major}.{p.minor}")
    torch.backends.cudnn.benchmark = True

USE_AMP = (DEVICE == "cuda")
NUM_WORKERS = 0 if platform.system() == "Windows" else 4
SEED = 42
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()

# --- spiking backend ---
try:
    from spikingjelly.activation_based import neuron, functional, surrogate, layer
    HAS_SPIKINGJELLY = True
except Exception as e:
    neuron = functional = surrogate = layer = None
    HAS_SPIKINGJELLY = False
    print("\nSpikingJelly import failed:", type(e).__name__, e)

# The reference benchmark is spiking. Flip this only if you accept the deviation.
ALLOW_ANN_FALLBACK = False

print("\nSpikingJelly available :", HAS_SPIKINGJELLY)
print("ANN fallback allowed   :", ALLOW_ANN_FALLBACK)
if not HAS_SPIKINGJELLY and not ALLOW_ANN_FALLBACK:
    print("\nInstall it before CELL 11:   pip install spikingjelly")
    print("(see the appendix at the end of this notebook)")
print("AMP:", USE_AMP, "| workers:", NUM_WORKERS, "| seed:", SEED, "| run:", RUN_ID)

ENVIRONMENT
Python      : 3.11.15
Platform    : Windows-10-10.0.26200-SP0
PyTorch     : 2.7.1+cu126
NumPy       : 1.26.4
SpikingJelly: installed
scikit-learn: 1.5.2
pandas      : 2.2.3
timm        : 1.0.28

DEVICE      : cuda
CUDA runtime: 12.6
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU | 8.0 GB | SM 8.9

SpikingJelly available : True
ANN fallback allowed   : False
AMP: True | workers: 0 | seed: 42 | run: 20260824_132146


---
# CELL 2 — UCF-Crime-DVS path configuration

The 116 GB dataset stays exactly where it is. Everything this notebook writes goes under
`G:\Sentrix\training` and `G:\Sentrix\backend\models\v2_real\ucf_crime_dvs`.

In [15]:
UCF_DVS_ROOT = Path(
    r"G:\event_frame_duration533326\event_frame_duration533326"
)
UCF_DVS_SPLIT_DIR = Path(
    r"G:\Sentrix\training\ucf_crime_dvs"
)
UCF_DVS_TRAIN_SPLIT = UCF_DVS_SPLIT_DIR / "train_split.txt"
UCF_DVS_TEST_SPLIT  = UCF_DVS_SPLIT_DIR / "test_split.txt"
UCF_DVS_GT_NPY      = UCF_DVS_SPLIT_DIR / "gt-ucf.npy"
UCF_DVS_GT_PICKLE   = UCF_DVS_SPLIT_DIR / "gt-ucf-dic.pickle"

UCF_DVS_RUN_DIR = Path(
    r"G:\Sentrix\training\runs_ucf_crime_dvs"
)
UCF_DVS_MODEL_DIR = Path(
    r"G:\Sentrix\backend\models\v2_real\ucf_crime_dvs"
)
UCF_DVS_CACHE_DIR = Path(
    r"G:\Sentrix\training\cache_ucf_crime_dvs"
)

for d in [UCF_DVS_SPLIT_DIR, UCF_DVS_RUN_DIR, UCF_DVS_MODEL_DIR, UCF_DVS_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

UCF_DVS_MODEL     = UCF_DVS_MODEL_DIR / "ucf_crime_dvs_anomaly_v1.pt"
UCF_DVS_METADATA  = UCF_DVS_MODEL_DIR / "ucf_crime_dvs_anomaly_v1_metadata.json"
UCF_DVS_PREDS_CSV = UCF_DVS_RUN_DIR / "sentrix_dvs_predictions.csv"

# The dataset directory is READ-ONLY for this notebook.
DATASET_READ_ONLY = True

print("=" * 64)
print("UCF-CRIME-DVS CONFIGURATION")
print("=" * 64)
print("RUN_ID      :", RUN_ID)
print("dataset     :", UCF_DVS_ROOT, "| exists:", UCF_DVS_ROOT.exists())
print("splits      :", UCF_DVS_SPLIT_DIR)
print("runs        :", UCF_DVS_RUN_DIR)
print("cache       :", UCF_DVS_CACHE_DIR)
print("model out   :", UCF_DVS_MODEL)
print("dataset is read-only:", DATASET_READ_ONLY)

if not UCF_DVS_ROOT.exists():
    raise RuntimeError(
        f"Dataset root not found: {UCF_DVS_ROOT}\n"
        "Fix the path above. Nothing will be generated to work around a missing dataset."
    )

UCF-CRIME-DVS CONFIGURATION
RUN_ID      : 20260824_132146
dataset     : G:\event_frame_duration533326\event_frame_duration533326 | exists: True
splits      : G:\Sentrix\training\ucf_crime_dvs
runs        : G:\Sentrix\training\runs_ucf_crime_dvs
cache       : G:\Sentrix\training\cache_ucf_crime_dvs
model out   : G:\Sentrix\backend\models\v2_real\ucf_crime_dvs\ucf_crime_dvs_anomaly_v1.pt
dataset is read-only: True


---
# CELL 3 — NPZ inventory

Counts the real `.npz` files per category. **No file is opened, moved, or written here.**

In [16]:
import re
import difflib

UCF_DVS_CATEGORIES = [
    "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion", "Fighting",
    "Normal_Videos", "RoadAccidents", "Robbery", "Shooting", "Shoplifting",
    "Stealing", "Vandalism",
]

# Known spelling variants shipped with the dataset. "Assualt" is a transposition
# in the official release, not a mistake on your disk.
CATEGORY_DIR_ALIASES = {
    "Assault":       ["Assualt", "Assult", "Asault"],
    "Normal_Videos": ["Normal_Videos_event", "NormalVideos", "Normal", "Testing_Normal_Videos"],
    "RoadAccidents": ["Road_Accidents", "RoadAccident", "Roadaccidents"],
    "Shoplifting":   ["ShopLifting", "Shop_lifting"],
}


def _norm_dir(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


_DIRS = [p for p in UCF_DVS_ROOT.iterdir() if p.is_dir()]
_DIR_BY_KEY = {_norm_dir(p.name): p for p in _DIRS}
_CLAIMED = set()


def _dir_for(cat):
    # 1 exact  2 case/underscore-insensitive  3 known alias  4 fuzzy  5 anagram (typos)
    key = _norm_dir(cat)
    exact = UCF_DVS_ROOT / cat
    if exact.exists() and exact not in _CLAIMED:
        return exact
    if key in _DIR_BY_KEY and _DIR_BY_KEY[key] not in _CLAIMED:
        return _DIR_BY_KEY[key]
    for alias in CATEGORY_DIR_ALIASES.get(cat, []):
        ak = _norm_dir(alias)
        if ak in _DIR_BY_KEY and _DIR_BY_KEY[ak] not in _CLAIMED:
            return _DIR_BY_KEY[ak]
    for m in difflib.get_close_matches(key, list(_DIR_BY_KEY), n=3, cutoff=0.8):
        if _DIR_BY_KEY[m] not in _CLAIMED:
            return _DIR_BY_KEY[m]
    target = "".join(sorted(key))
    for kk, pp in _DIR_BY_KEY.items():
        if "".join(sorted(kk)) == target and pp not in _CLAIMED:
            return pp
    return exact


INVENTORY, TOTAL_BYTES, all_npz = {}, 0, []
print("=" * 78)
print("UCF-CRIME-DVS NPZ INVENTORY")
print("=" * 78)
print("root:", UCF_DVS_ROOT)
print(f"\n{'category':18}{'files':>8}{'size (GB)':>12}  directory on disk")
print("-" * 78)

for cat in UCF_DVS_CATEGORIES:
    d = _dir_for(cat)
    if d.exists():
        _CLAIMED.add(d)
    files = sorted(d.rglob("*.npz")) if d.exists() else []
    size = sum(f.stat().st_size for f in files)
    TOTAL_BYTES += size
    INVENTORY[cat] = {"dir": str(d), "dir_name": d.name, "n_files": len(files),
                      "bytes": size, "exists": d.exists(),
                      "name_differs": d.exists() and _norm_dir(d.name) != _norm_dir(cat)}
    all_npz += [(f, cat) for f in files]
    note = ""
    if not d.exists():
        note = "   <-- MISSING"
    elif INVENTORY[cat]["name_differs"]:
        note = "   <-- folder name differs (mapped)"
    print(f"{cat:18}{len(files):>8}{size/1024**3:>12.2f}  {d.name}{note}")

print("-" * 78)
print(f"{'TOTAL':18}{len(all_npz):>8}{TOTAL_BYTES/1024**3:>12.2f}")

# any directory on disk that no category claimed -> would silently lose data
orphans = [p.name for p in _DIRS if p not in _CLAIMED]
if orphans:
    print("\nUNMAPPED DIRECTORIES (their files are NOT in all_npz):")
    for o in orphans:
        n_files = len(list((UCF_DVS_ROOT / o).rglob("*.npz")))
        print(f"   {o}  ({n_files} npz)")
    print("Add the correct spelling to CATEGORY_DIR_ALIASES above and rerun this cell.")
else:
    print("\nEvery directory on disk is mapped to a category.")

missing = [c for c, v in INVENTORY.items() if not v["exists"] or v["n_files"] == 0]
if missing:
    print("\nCATEGORIES WITH NO FILES:", ", ".join(missing))
if not all_npz:
    raise RuntimeError("No .npz files found. Check UCF_DVS_ROOT in CELL 2.")

n_anom = sum(v["n_files"] for c, v in INVENTORY.items() if c != "Normal_Videos")
n_norm = INVENTORY.get("Normal_Videos", {}).get("n_files", 0)
print(f"\nanomalous videos: {n_anom}")
print(f"normal videos   : {n_norm}")

(UCF_DVS_RUN_DIR / f"inventory_{RUN_ID}.json").write_text(
    json.dumps(INVENTORY, indent=2), encoding="utf-8")
print("\nsaved:", UCF_DVS_RUN_DIR / f"inventory_{RUN_ID}.json")

UCF-CRIME-DVS NPZ INVENTORY
root: G:\event_frame_duration533326\event_frame_duration533326

category             files   size (GB)  directory on disk
------------------------------------------------------------------------------
Abuse                   50        1.57  Abuse
Arrest                  53        2.62  Arrest
Arson                   57        2.49  Arson
Assault                 50        1.18  Assualt   <-- folder name differs (mapped)
Burglary               100        3.14  Burglary
Explosion               54        2.02  Explosion
Fighting                53        2.44  Fighting
Normal_Videos         1271       89.67  Normal_Videos
RoadAccidents          150        2.43  RoadAccidents
Robbery                151        4.46  Robbery
Shooting                50        1.29  Shooting
Shoplifting             53        3.37  Shoplifting
Stealing               100        2.92  Stealing
Vandalism               50        0.99  Vandalism
---------------------------------------------

---
# CELL 4 — Inspect NPZ structure

Opens a handful of real files and reports the arrays inside. Everything downstream adapts to
what is actually found — nothing about the layout is hard-coded.

Two layouts are supported:

| Layout | Keys | Meaning |
|---|---|---|
| **integrated frames** | `frames` (or a single array) | `[T, 2, H, W]` event frames already integrated by fixed duration |
| **raw events** | `t`, `x`, `y`, `p` | event stream, integrated into frames by CELL 9 |

In [17]:
import zipfile
import numpy.lib.format as npf

# ---------------------------------------------------------------
# Memory-safe NPZ access.
# These files are (T, 2, 720, 1280) float64 -> up to 10 GB decompressed.
# Never call np.load(...)[key] on one. Read the header, stream the frames.
# ---------------------------------------------------------------
def _npy_header(f):
    version = npf.read_magic(f)
    if version == (1, 0):
        return npf.read_array_header_1_0(f)
    if version == (2, 0):
        return npf.read_array_header_2_0(f)
    return npf._read_array_header(f, version)


def npz_header_info(path):
    # shape/dtype of every array WITHOUT decompressing any data
    info = {}
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not name.endswith(".npy"):
                continue
            with z.open(name) as f:
                shape, fortran, dtype = _npy_header(f)
            info[name[:-4]] = {"shape": tuple(int(s) for s in shape),
                               "dtype": str(dtype), "fortran_order": bool(fortran)}
    return info


def _read_exact(f, n):
    chunks, got = [], 0
    while got < n:
        b = f.read(n - got)
        if not b:
            raise EOFError("unexpected end of npy stream")
        chunks.append(b); got += len(b)
    return b"".join(chunks)


def _skip(f, n):
    while n > 0:
        b = f.read(min(n, 1 << 22))
        if not b:
            raise EOFError("unexpected end while skipping")
        n -= len(b)


def iter_frames_by_index(path, key, indices):
    # Yields (index, frame) for the requested frames only, in ascending order.
    # One frame of this dataset is 2*720*1280*8 = 14.7 MB - the whole array is not.
    idxs = sorted(set(int(i) for i in indices))
    with zipfile.ZipFile(path) as z:
        with z.open(key + ".npy") as f:
            shape, fortran, dtype = _npy_header(f)
            if fortran:
                raise RuntimeError("fortran-ordered npy is not supported")
            frame_shape = tuple(int(s) for s in shape[1:])
            nbytes = int(np.prod(frame_shape)) * dtype.itemsize
            pos = 0
            for i in idxs:
                if i >= shape[0]:
                    break
                if i > pos:
                    _skip(f, (i - pos) * nbytes)
                buf = _read_exact(f, nbytes)
                yield i, np.frombuffer(buf, dtype=dtype).reshape(frame_shape).copy()
                pos = i + 1


def describe_npz(path, max_keys=12):
    # header only + the FIRST frame for a value range. Nothing else is decompressed.
    out = {"path": str(path), "size_mb": round(path.stat().st_size / 1024**2, 2), "arrays": {}}
    hdr = npz_header_info(path)
    for k in list(hdr.keys())[:max_keys]:
        entry = {"shape": hdr[k]["shape"], "dtype": hdr[k]["dtype"],
                 "min": None, "max": None, "range_from": "header only"}
        if len(hdr[k]["shape"]) >= 3 and np.dtype(hdr[k]["dtype"]).kind in "iuf":
            try:
                _, f0 = next(iter_frames_by_index(path, k, [0]))
                entry["min"], entry["max"] = float(f0.min()), float(f0.max())
                entry["range_from"] = "frame 0"
            except Exception:
                pass
        out["arrays"][k] = entry
    return out


set_seed()
probe = random.sample(all_npz, min(5, len(all_npz)))

print("=" * 64)
print("NPZ STRUCTURE PROBE  (header + first frame only)")
print("=" * 64)
descs = []
for f, cat in probe:
    d = describe_npz(f)
    d["category"] = cat
    descs.append(d)
    print(f"\n{cat} :: {f.name}   ({d['size_mb']} MB on disk)")
    for k, v in d["arrays"].items():
        print(f"   {k:10} shape={str(v['shape']):26} dtype={v['dtype']:8} "
              f"range=[{v['min']}, {v['max']}] ({v['range_from']})")

keysets = [set(d["arrays"].keys()) for d in descs]
common = set.intersection(*keysets) if keysets else set()

if {"t", "x", "y", "p"} <= common or {"t", "x", "y", "pol"} <= common:
    NPZ_LAYOUT = "raw_events"
elif "frames" in common:
    NPZ_LAYOUT = "frames"
elif len(common) == 1:
    NPZ_LAYOUT = "single_array"
else:
    NPZ_LAYOUT = "unknown"

FRAME_KEY = ("frames" if NPZ_LAYOUT == "frames"
             else (list(common)[0] if NPZ_LAYOUT == "single_array" else None))

EVENT_H = EVENT_W = None
POLARITY_AXIS = None
if NPZ_LAYOUT in ("frames", "single_array"):
    shp = descs[0]["arrays"][FRAME_KEY]["shape"]
    if len(shp) == 4 and shp[1] == 2:
        POLARITY_AXIS, EVENT_H, EVENT_W = 1, shp[2], shp[3]
    elif len(shp) == 4 and shp[3] == 2:
        POLARITY_AXIS, EVENT_H, EVENT_W = 3, shp[1], shp[2]
    elif len(shp) == 3:
        POLARITY_AXIS, EVENT_H, EVENT_W = None, shp[1], shp[2]

print("\n" + "=" * 64)
print("DETECTED LAYOUT")
print("=" * 64)
print("layout        :", NPZ_LAYOUT)
print("frame key     :", FRAME_KEY)
print("polarity axis :", POLARITY_AXIS)
print("native H x W  :", EVENT_H, "x", EVENT_W)
if NPZ_LAYOUT in ("frames", "single_array"):
    dt = np.dtype(descs[0]["arrays"][FRAME_KEY]["dtype"])
    Ts = [d["arrays"][FRAME_KEY]["shape"][0] for d in descs]
    per_frame = 2 * EVENT_H * EVENT_W * dt.itemsize / 1024**2
    print("frames per file (probe):", Ts)
    print(f"one frame decompressed : {per_frame:.1f} MB")
    print(f"WHOLE array would be   : {max(Ts) * per_frame / 1024:.1f} GB  <-- never load it")

if NPZ_LAYOUT == "unknown":
    raise RuntimeError(
        "Could not recognise the NPZ layout. Keys seen: " + str(keysets) +
        "\nAdapt load_event_frames() in CELL 8 to this layout before continuing."
    )

(UCF_DVS_RUN_DIR / f"npz_probe_{RUN_ID}.json").write_text(
    json.dumps({"layout": NPZ_LAYOUT, "frame_key": FRAME_KEY,
                "polarity_axis": POLARITY_AXIS, "H": EVENT_H, "W": EVENT_W,
                "probe": descs}, indent=2, default=str), encoding="utf-8")

NPZ STRUCTURE PROBE  (header + first frame only)

Normal_Videos :: Normal_Videos058_x264_61.npz   (4.48 MB on disk)
   frames     shape=(61, 2, 720, 1280)         dtype=float64  range=[0.0, 35.0] (frame 0)

Arrest :: Arrest051_x264_669.npz   (78.02 MB on disk)
   frames     shape=(669, 2, 720, 1280)        dtype=float64  range=[0.0, 71.0] (frame 0)

Normal_Videos :: Normal_Videos560_x264_1137.npz   (203.22 MB on disk)
   frames     shape=(1137, 2, 720, 1280)       dtype=float64  range=[0.0, 92.0] (frame 0)

Normal_Videos :: Normal_Videos491_01_x264_377.npz   (125.8 MB on disk)
   frames     shape=(377, 2, 720, 1280)        dtype=float64  range=[0.0, 55.0] (frame 0)

Normal_Videos :: Normal_Videos432_x264_223.npz   (41.74 MB on disk)
   frames     shape=(223, 2, 720, 1280)        dtype=float64  range=[0.0, 57.0] (frame 0)

DETECTED LAYOUT
layout        : frames
frame key     : frames
polarity axis : 1
native H x W  : 720 x 1280
frames per file (probe): [61, 669, 1137, 377, 223]
one fram

2390

---
# CELL 5 — Map the 14 UCF-Crime-DVS categories

Two label spaces, both derived from **real folder names** — nothing is invented:

- **binary anomaly label** (the supervision that drives the MIL scorer):
  `Normal_Videos → 0`, every other category `→ 1`
- **category label** (auxiliary head only): 14-way

In [18]:
CATEGORY_TO_IDX = {c: i for i, c in enumerate(UCF_DVS_CATEGORIES)}
IDX_TO_CATEGORY = {i: c for c, i in CATEGORY_TO_IDX.items()}
NORMAL_CATEGORY = "Normal_Videos"
NORMAL_IDX = CATEGORY_TO_IDX[NORMAL_CATEGORY]
ANOMALY_CATEGORIES = [c for c in UCF_DVS_CATEGORIES if c != NORMAL_CATEGORY]

def binary_label(category):
    return 0 if category == NORMAL_CATEGORY else 1

print("=" * 64)
print("LABEL SPACES")
print("=" * 64)
print(f"{'idx':>4}  {'category':18}{'binary':>8}{'files':>8}")
print("-" * 42)
for c in UCF_DVS_CATEGORIES:
    print(f"{CATEGORY_TO_IDX[c]:>4}  {c:18}{binary_label(c):>8}{INVENTORY[c]['n_files']:>8}")

print(f"\nnormal category : {NORMAL_CATEGORY} (idx {NORMAL_IDX})")
print(f"anomaly classes : {len(ANOMALY_CATEGORIES)}")
print("\nSENTRIX contract: only the BINARY anomaly score is consumed by TCI.")
print("The 14-way head is auxiliary context and is weakly supervised.")

LABEL SPACES
 idx  category            binary   files
------------------------------------------
   0  Abuse                    1      50
   1  Arrest                   1      53
   2  Arson                    1      57
   3  Assault                  1      50
   4  Burglary                 1     100
   5  Explosion                1      54
   6  Fighting                 1      53
   7  Normal_Videos            0    1271
   8  RoadAccidents            1     150
   9  Robbery                  1     151
  10  Shooting                 1      50
  11  Shoplifting              1      53
  12  Stealing                 1     100
  13  Vandalism                1      50

normal category : Normal_Videos (idx 7)
anomaly classes : 13

SENTRIX contract: only the BINARY anomaly score is consumed by TCI.
The 14-way head is auxiliary context and is weakly supervised.


---
# CELL 6 — Load the official `train_split.txt` / `test_split.txt`

Order of preference:

1. local copies in `G:\Sentrix\training\ucf_crime_dvs\`
2. downloaded from the official repository (several candidate paths are tried) and saved locally
3. **stop** — the notebook refuses to invent a split, because a home-made partition makes the
   numbers incomparable to the published benchmark

If you must proceed without the official files, set `ALLOW_DERIVED_SPLIT = True` below and rerun;
every result will then be stamped `NOT COMPARABLE TO PUBLISHED BENCHMARK`.

In [19]:
ALLOW_DERIVED_SPLIT = False          # keep False unless you accept non-comparable numbers
SPLIT_DOWNLOAD_ENABLED = True        # try to fetch the official files if missing

OFFICIAL_REPO = "https://github.com/YBQian-Roy/UCF-Crime-DVS"
RAW_BASES = [
    "https://raw.githubusercontent.com/YBQian-Roy/UCF-Crime-DVS/main/",
    "https://raw.githubusercontent.com/YBQian-Roy/UCF-Crime-DVS/master/",
]
SPLIT_CANDIDATE_PATHS = [
    "{f}", "list/{f}", "lists/{f}", "splits/{f}", "data/{f}",
    "MSF/list/{f}", "feature_extractor/list/{f}",
]
WANTED = {
    "train_split.txt": UCF_DVS_TRAIN_SPLIT,
    "test_split.txt":  UCF_DVS_TEST_SPLIT,
    "gt-ucf.npy":      UCF_DVS_GT_NPY,
    "gt-ucf-dic.pickle": UCF_DVS_GT_PICKLE,
}

def try_download(fname, dest: Path):
    import urllib.request
    for base in RAW_BASES:
        for tmpl in SPLIT_CANDIDATE_PATHS:
            url = base + tmpl.format(f=fname)
            try:
                with urllib.request.urlopen(url, timeout=20) as r:
                    if r.status == 200:
                        data = r.read()
                        if len(data) > 0:
                            dest.write_bytes(data)
                            print(f"   downloaded {fname} <- {url}  ({len(data)} bytes)")
                            return True
            except Exception:
                continue
    return False


print("=" * 64)
print("OFFICIAL SPLIT / GROUND-TRUTH FILES")
print("=" * 64)
for fname, dest in WANTED.items():
    if dest.exists():
        print(f"   present  {fname:20} {dest}  ({dest.stat().st_size} bytes)")
    elif SPLIT_DOWNLOAD_ENABLED:
        print(f"   missing  {fname:20} - trying the official repository...")
        if not try_download(fname, dest):
            print(f"   FAILED   {fname:20} - could not be fetched automatically")
    else:
        print(f"   missing  {fname:20} (download disabled)")

HAVE_OFFICIAL_SPLIT = UCF_DVS_TRAIN_SPLIT.exists() and UCF_DVS_TEST_SPLIT.exists()
HAVE_FRAME_GT = UCF_DVS_GT_NPY.exists() or UCF_DVS_GT_PICKLE.exists()

print("\nofficial split available :", HAVE_OFFICIAL_SPLIT)
print("frame-level GT available :", HAVE_FRAME_GT)

if not HAVE_OFFICIAL_SPLIT and not ALLOW_DERIVED_SPLIT:
    raise RuntimeError(
        "Official train_split.txt / test_split.txt not found and could not be downloaded.\n\n"
        f"Download them manually from {OFFICIAL_REPO} and place them in:\n"
        f"   {UCF_DVS_SPLIT_DIR}\n\n"
        "A home-made split would make these results incomparable to the published benchmark, "
        "so this notebook stops rather than inventing one.\n"
        "If you accept that trade-off, set ALLOW_DERIVED_SPLIT = True in this cell and rerun."
    )


def parse_split_file(path: Path):
    # Each line names a video; extra whitespace-separated fields (labels, frame counts)
    # are ignored. Returns a list of normalised stems.
    stems = []
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        token = line.split()[0].replace("\\", "/")
        stem = Path(token).name
        for suf in [".npz", ".npy", ".mp4", ".avi", "_x264", ".txt"]:
            if stem.endswith(suf):
                stem = stem[: -len(suf)]
        stems.append(stem)
    return stems


TRAIN_STEMS, TEST_STEMS = [], []
if HAVE_OFFICIAL_SPLIT:
    TRAIN_STEMS = parse_split_file(UCF_DVS_TRAIN_SPLIT)
    TEST_STEMS = parse_split_file(UCF_DVS_TEST_SPLIT)
    print(f"\nofficial train entries: {len(TRAIN_STEMS)}")
    print(f"official test entries : {len(TEST_STEMS)}")
    print("first 5 train:", TRAIN_STEMS[:5])
    print("first 5 test :", TEST_STEMS[:5])
    overlap = set(TRAIN_STEMS) & set(TEST_STEMS)
    print("train/test overlap    :", len(overlap), "<-- must be 0" if overlap else "")
else:
    print("\nPROCEEDING WITH A DERIVED SPLIT — results are NOT comparable to the benchmark.")

OFFICIAL SPLIT / GROUND-TRUTH FILES
   present  train_split.txt      G:\Sentrix\training\ucf_crime_dvs\train_split.txt  (38057 bytes)
   present  test_split.txt       G:\Sentrix\training\ucf_crime_dvs\test_split.txt  (7143 bytes)
   present  gt-ucf.npy           G:\Sentrix\training\ucf_crime_dvs\gt-ucf.npy  (8913280 bytes)
   present  gt-ucf-dic.pickle    G:\Sentrix\training\ucf_crime_dvs\gt-ucf-dic.pickle  (10039386 bytes)

official split available : True
frame-level GT available : True

official train entries: 1610
official test entries : 290
first 5 train: ['Abuse001', 'Abuse002', 'Abuse003', 'Abuse004', 'Abuse005']
first 5 test : ['Abuse028', 'Abuse030', 'Arrest001', 'Arrest007', 'Arrest024']
train/test overlap    : 0 


---
# CELL 7 — Build train / validation / test samples

- **test** = the official test split, untouched
- **validation** = a seeded, stratified slice carved out of the official *train* split
  (the benchmark has no separate val set; the test split is never used for model selection)
- every sample carries its real `.npz` path, category and binary label

In [20]:
import re
VAL_FRACTION = 0.15


def stem_keys(s):
    # All plausible normalised forms of a video id, most specific first.
    # Handles:  Abuse051_x264_669.npz   Normal_Videos_110_x264   Arrest051_x264.mp4   Abuse051
    s = Path(str(s).replace("\\", "/")).name
    s = re.sub(r"\.(npz|npy|mp4|avi|txt)$", "", s, flags=re.I)
    cands = [s]
    cands.append(re.sub(r"(_x264)_\d+$", r"\1", s, flags=re.I))   # drop frame-count tail
    cands.append(re.sub(r"_x264$", "", cands[-1], flags=re.I))    # drop _x264
    cands.append(re.sub(r"_\d+$", "", s))                         # generic numeric tail
    cands.append(re.sub(r"_x264.*$", "", s, flags=re.I))          # everything from _x264 on
    out = []
    for c in cands:
        k = re.sub(r"[^a-z0-9]", "", c.lower())    # ignore underscores/case entirely
        if k and k not in out:
            out.append(k)
    return out


def norm_stem(s):
    # canonical key (also used by the frame-level GT alignment in CELL 14)
    ks = stem_keys(s)
    return ks[-1] if ks else str(s).lower()


# index every real file under ALL of its key variants
BY_STEM = {}
for f, cat in all_npz:
    for k in stem_keys(f.stem):
        BY_STEM.setdefault(k, [])
        if (f, cat) not in BY_STEM[k]:
            BY_STEM[k].append((f, cat))


def resolve(stems):
    items, misses, seen = [], [], set()
    for s in stems:
        hits = None
        for k in stem_keys(s):
            if k in BY_STEM:
                hits = BY_STEM[k]
                break
        if not hits:
            misses.append(s)
            continue
        for f, cat in hits:                      # keep every chunk of the same video
            rp = str(f).lower()
            if rp in seen:
                continue
            seen.add(rp)
            items.append({"path": f, "stem": f.stem, "category": cat,
                          "cat_idx": CATEGORY_TO_IDX[cat], "label": binary_label(cat)})
    return items, misses


# ---------------- diagnostics before anything can fail silently ----------------
print("=" * 64)
print("STEM MATCHING")
print("=" * 64)
_disk_sample = [f.stem for f, _ in all_npz[:4]]
print("on disk  :", _disk_sample)
print("keys     :", [stem_keys(s)[-1] for s in _disk_sample])
if HAVE_OFFICIAL_SPLIT:
    _split_sample = (TRAIN_STEMS + TEST_STEMS)[:4]
    print("in split :", _split_sample)
    print("keys     :", [stem_keys(s)[-1] for s in _split_sample])

if HAVE_OFFICIAL_SPLIT:
    train_pool, train_miss = resolve(TRAIN_STEMS)
    test_items, test_miss = resolve(TEST_STEMS)
    print(f"\nresolved train: {len(train_pool)}/{len(TRAIN_STEMS)}  (missing {len(train_miss)})")
    print(f"resolved test : {len(test_items)}/{len(TEST_STEMS)}  (missing {len(test_miss)})")
    for s in (train_miss + test_miss)[:10]:
        print("   no .npz on disk for:", s)

    if not train_pool or not test_items:
        raise RuntimeError(
            "Still no matches between the split files and the files on disk.\n"
            f"  disk stem : {_disk_sample[0] if _disk_sample else 'NONE'}\n"
            f"  disk keys : {stem_keys(_disk_sample[0]) if _disk_sample else []}\n"
            f"  split line: {(TRAIN_STEMS[:1] or ['NONE'])[0]}\n"
            f"  split keys: {stem_keys(TRAIN_STEMS[0]) if TRAIN_STEMS else []}\n"
            "Extend stem_keys() so one form of each appears in both lists."
        )
    SPLIT_SOURCE = "official train_split.txt / test_split.txt"
else:
    from sklearn.model_selection import train_test_split as _tts
    allit = [{"path": f, "stem": f.stem, "category": c,
              "cat_idx": CATEGORY_TO_IDX[c], "label": binary_label(c)} for f, c in all_npz]
    train_pool, test_items = _tts(allit, test_size=0.2, random_state=SEED,
                                  stratify=[a["cat_idx"] for a in allit])
    SPLIT_SOURCE = "DERIVED in-notebook split - NOT COMPARABLE TO PUBLISHED BENCHMARK"

if not train_pool or not test_items:
    raise RuntimeError("Empty train or test set after resolution - check the split files "
                       "and the dataset root.")

from sklearn.model_selection import train_test_split
strat = [a["cat_idx"] for a in train_pool]
try:
    train_items, val_items = train_test_split(
        train_pool, test_size=VAL_FRACTION, random_state=SEED, stratify=strat)
except ValueError:      # a category with a single member
    train_items, val_items = train_test_split(
        train_pool, test_size=VAL_FRACTION, random_state=SEED,
        stratify=[a["label"] for a in train_pool])

# guarantee disjointness by resolved path
_tr = {str(a["path"]).lower() for a in train_items}
val_items = [a for a in val_items if str(a["path"]).lower() not in _tr]
_trva = _tr | {str(a["path"]).lower() for a in val_items}
test_items = [a for a in test_items if str(a["path"]).lower() not in _trva]


def dist(items):
    d = {}
    for a in items:
        d[a["category"]] = d.get(a["category"], 0) + 1
    return d


print("\n" + "=" * 64)
print("SPLITS")
print("=" * 64)
print("source:", SPLIT_SOURCE)
for nm, it in [("train", train_items), ("val", val_items), ("test", test_items)]:
    n_anom = sum(a["label"] for a in it)
    print(f"\n{nm:5}: {len(it):>5} videos   anomalous={n_anom}  normal={len(it)-n_anom}")
    for c, n in sorted(dist(it).items()):
        print(f"        {c:18}{n:>5}")

import pandas as pd
for nm, it in [("train", train_items), ("val", val_items), ("test", test_items)]:
    pd.DataFrame([{"source_path": str(a["path"]), "stem": a["stem"],
                   "category": a["category"], "cat_idx": a["cat_idx"],
                   "binary_label": a["label"]} for a in it]).to_csv(
        UCF_DVS_RUN_DIR / f"sample_manifest_{nm}.csv", index=False)
print("\nmanifests written to:", UCF_DVS_RUN_DIR)

STEM MATCHING
on disk  : ['Abuse001_x264_171', 'Abuse002_x264_53', 'Abuse003_x264_229', 'Abuse004_x264_1048']
keys     : ['abuse001', 'abuse002', 'abuse003', 'abuse004']
in split : ['Abuse001', 'Abuse002', 'Abuse003', 'Abuse004']
keys     : ['abuse001', 'abuse002', 'abuse003', 'abuse004']

resolved train: 1549/1610  (missing 77)
resolved test : 287/290  (missing 3)
   no .npz on disk for: Arrest012
   no .npz on disk for: Arrest047
   no .npz on disk for: Arson019
   no .npz on disk for: Explosion046
   no .npz on disk for: Fighting008
   no .npz on disk for: Fighting041
   no .npz on disk for: Fighting050
   no .npz on disk for: Robbery014
   no .npz on disk for: Shoplifting014
   no .npz on disk for: Shoplifting040

SPLITS
source: official train_split.txt / test_split.txt

train:  1316 videos   anomalous=680  normal=636
        Abuse                41
        Arrest               36
        Arson                34
        Assault              40
        Burglary             74
      

---
# CELL 8 — Load NPZ event data

One function, adapting to whichever layout CELL 4 detected. It returns
`[T, 2, H, W]` float32 event frames — **polarity is preserved as two channels**, which is the
whole point of event data and the thing an image pipeline would destroy.

---
# CELL 9 — Event preprocessing

| Step | What happens | Why |
|---|---|---|
| polarity | kept as 2 channels (ON / OFF) | the defining property of DVS data |
| spatial | **max**-pool to `TARGET_HW` | events are sparse; average-pooling erases them |
| temporal | split into `N_SEGMENTS` MIL segments, `FRAMES_PER_SEG` SNN timesteps each | video-level supervision needs instances |
| value | clip to `EVENT_CLIP` then scale to `[0,1]` | tames hot pixels without discarding count information |

No augmentation is applied that would invent events — only horizontal flip and segment jitter
of the real stream, both switchable.

In [21]:
# ===== CELL 8 + 9 (merged): streaming loader + preprocessing =====
TARGET_HW       = 128
N_SEGMENTS      = 32
FRAMES_PER_SEG  = 4
EVENT_CLIP      = 32.0    # raised from 8: your raw counts reach ~230
AUG_HFLIP       = True
AUG_SEG_JITTER  = True


def event_frame_count(path):
    # number of frames WITHOUT decompressing anything
    return npz_header_info(Path(path))[FRAME_KEY]["shape"][0]


def _raw_events_to_frames(t, x, y, p, n_frames, H, W):
    t = np.asarray(t).ravel(); x = np.asarray(x).ravel().astype(np.int64)
    y = np.asarray(y).ravel().astype(np.int64); p = np.asarray(p).ravel()
    n = len(t)
    if n == 0:
        return np.zeros((n_frames, 2, H, W), np.float32)
    order = np.argsort(t, kind="stable")
    x, y, p = x[order], y[order], p[order]
    pol = (p > 0).astype(np.int64)
    edges = np.linspace(0, n, n_frames + 1).astype(np.int64)
    frames = np.zeros((n_frames, 2, H * W), np.float32)
    flat = np.clip(y, 0, H - 1) * W + np.clip(x, 0, W - 1)
    for i in range(n_frames):
        s, e = edges[i], edges[i + 1]
        if e <= s:
            continue
        for pv in (0, 1):
            m = pol[s:e] == pv
            if m.any():
                frames[i, pv] += np.bincount(flat[s:e][m], minlength=H * W).astype(np.float32)
    return frames.reshape(n_frames, 2, H, W)


def temporal_segments(T, n_segments=N_SEGMENTS, frames_per_seg=FRAMES_PER_SEG, jitter=False):
    edges = np.linspace(0, T, n_segments + 1)
    idx = np.zeros((n_segments, frames_per_seg), dtype=np.int64)
    for s in range(n_segments):
        lo, hi = edges[s], edges[s + 1]
        if hi - lo < 1e-6:
            picks = np.full(frames_per_seg, min(int(lo), T - 1))
        else:
            base = np.linspace(lo, hi - 1e-3, frames_per_seg)
            if jitter:
                span = max((hi - lo) / frames_per_seg, 1.0)
                base = base + np.random.uniform(-0.5, 0.5, frames_per_seg) * span
            picks = np.clip(base.astype(np.int64), 0, T - 1)
        idx[s] = picks
    return idx


def _pool_frame(fr, hw=TARGET_HW):
    # one REAL frame -> [2, hw, hw] float32, MAX pooled (sparse-event friendly)
    t = torch.from_numpy(np.ascontiguousarray(fr, dtype=np.float32))
    if t.ndim == 2:
        t = torch.stack([t, torch.zeros_like(t)])
    return F.adaptive_max_pool2d(t.unsqueeze(0), (hw, hw))[0]


def preprocess_video(path, jitter=False, hflip=False):
    # REAL npz -> uint8 [N_SEGMENTS, FRAMES_PER_SEG, 2, TARGET_HW, TARGET_HW]
    # Streams ONLY the frames it needs; the full array is never materialised.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"event file missing on disk: {path}")

    if NPZ_LAYOUT == "raw_events":
        with np.load(path, allow_pickle=True) as d:
            pk = "p" if "p" in d else ("pol" if "pol" in d else "polarity")
            H = EVENT_H or int(np.max(d["y"])) + 1
            W = EVENT_W or int(np.max(d["x"])) + 1
            frames = _raw_events_to_frames(d["t"], d["x"], d["y"], d[pk],
                                           N_SEGMENTS * FRAMES_PER_SEG, H, W)
        pooled = [_pool_frame(f) for f in frames]
    else:
        T = event_frame_count(path)
        flat = temporal_segments(T, jitter=jitter).reshape(-1)
        cache = {i: _pool_frame(fr) for i, fr in iter_frames_by_index(path, FRAME_KEY, flat)}
        if not cache:
            raise RuntimeError(f"no frames could be read from {path.name}")
        last = cache[max(cache)]
        pooled = [cache.get(int(i), last) for i in flat]

    small = torch.stack(pooled)
    if hflip:
        small = torch.flip(small, dims=[-1])
    small = small.clamp(0, EVENT_CLIP) * (255.0 / EVENT_CLIP)
    return small.to(torch.uint8).reshape(N_SEGMENTS, FRAMES_PER_SEG, 2, TARGET_HW, TARGET_HW)


# ---- checks on real files ----
print("=" * 64)
print("STREAMING LOADER CHECK (real files)")
print("=" * 64)
for a in train_items[:3]:
    t0 = time.time()
    T = event_frame_count(a["path"])
    _, f0 = next(iter_frames_by_index(a["path"], FRAME_KEY, [0]))
    print(f"{a['category']:16} {a['stem'][:34]:36} T={T:>4}  frame{f0.shape} "
          f"{f0.dtype}  max={f0.max():.0f}  {time.time()-t0:.2f}s")

NATIVE_H, NATIVE_W = EVENT_H, EVENT_W
print(f"\nnative geometry: 2 x {NATIVE_H} x {NATIVE_W}  (T varies per video)")

print("\n" + "=" * 64)
print("PREPROCESSING CHECK")
print("=" * 64)
t0 = time.time()
_x = preprocess_video(train_items[0]["path"])
sat = (_x == 255).float().mean().item()
print("input :", train_items[0]["path"].name)
print("output:", tuple(_x.shape), _x.dtype,
      f"| {_x.numel()/1024**2:.2f} MB | {time.time()-t0:.1f}s")
print(f"occupancy (non-zero): {(_x > 0).float().mean().item():.4%}")
print(f"saturated at EVENT_CLIP={EVENT_CLIP:.0f}: {sat:.4%}"
      "   <-- raise EVENT_CLIP if this is more than a few percent")
print(f"\nper-video cache: {_x.numel()/1024**2:.2f} MB "
      f"| whole dataset approx {_x.numel()/1024**3*len(all_npz):.1f} GB")
del _x

STREAMING LOADER CHECK (real files)
RoadAccidents    RoadAccidents111_x264_76             T=  76  frame(2, 720, 1280) float64  max=54  0.03s
Stealing         Stealing066_x264_79                  T=  79  frame(2, 720, 1280) float64  max=68  0.04s
Normal_Videos    Normal_Videos829_x264_87             T=  87  frame(2, 720, 1280) float64  max=46  0.03s

native geometry: 2 x 720 x 1280  (T varies per video)

PREPROCESSING CHECK
input : RoadAccidents111_x264_76.npz
output: (32, 4, 2, 128, 128) torch.uint8 | 4.00 MB | 2.0s
occupancy (non-zero): 68.2425%
saturated at EVENT_CLIP=32: 0.0111%   <-- raise EVENT_CLIP if this is more than a few percent

per-video cache: 4.00 MB | whole dataset approx 8.8 GB


---
# CELL 10 — Event representation / tensor conversion (cache build)

Decompressing a multi-hundred-MB `.npz` on every epoch is the bottleneck, so each video is
preprocessed **once** into a small `.npy` under `G:\Sentrix\training\cache_ucf_crime_dvs`.
The cache is resumable — rerun the cell after an interruption and it skips what exists.

Set `BUILD_CACHE = False` to stream straight from the `.npz` files instead (much slower).
Set `MAX_VIDEOS_PER_SPLIT` to a small number for a quick end-to-end smoke run.

In [22]:
BUILD_CACHE = True
MAX_VIDEOS_PER_SPLIT = None      # e.g. 40 for a smoke run; None = everything
CACHE_TAG = f"hw{TARGET_HW}_n{N_SEGMENTS}_f{FRAMES_PER_SEG}_c{int(EVENT_CLIP)}"
CACHE_ROOT = UCF_DVS_CACHE_DIR / CACHE_TAG
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

if MAX_VIDEOS_PER_SPLIT:
    train_items = train_items[:MAX_VIDEOS_PER_SPLIT]
    val_items = val_items[:max(4, MAX_VIDEOS_PER_SPLIT // 4)]
    test_items = test_items[:max(4, MAX_VIDEOS_PER_SPLIT // 4)]
    print(f"SMOKE RUN: capped to {len(train_items)}/{len(val_items)}/{len(test_items)} videos\n")


def cache_path_for(item):
    return CACHE_ROOT / item["category"] / (item["stem"] + ".npy")


def build_cache(items, tag):
    done = skipped = failed = 0
    t0 = time.time()
    for i, a in enumerate(items, 1):
        cp = cache_path_for(a)
        if cp.exists() and cp.stat().st_size > 0:
            skipped += 1
            continue
        cp.parent.mkdir(parents=True, exist_ok=True)
        try:
            x = preprocess_video(a["path"])
            np.save(cp, x.numpy())
            done += 1
        except Exception as e:
            failed += 1
            print(f"   FAILED {a['stem']}: {type(e).__name__}: {e}")
        if i % 25 == 0 or i == len(items):
            el = time.time() - t0
            rate = i / max(el, 1e-6)
            print(f"   [{tag}] {i}/{len(items)}  built={done} cached={skipped} "
                  f"failed={failed}  {rate:.2f} vid/s  eta {(len(items)-i)/max(rate,1e-6)/60:.1f} min")
    return {"built": done, "reused": skipped, "failed": failed}


print("=" * 64)
print("TENSOR CACHE")
print("=" * 64)
print("cache root:", CACHE_ROOT)
print("tag       :", CACHE_TAG)

CACHE_STATS = {}
if BUILD_CACHE:
    for tag, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
        print(f"\nbuilding {tag} cache ({len(items)} videos)...")
        CACHE_STATS[tag] = build_cache(items, tag)
        print("  ", CACHE_STATS[tag])
    total = sum(f.stat().st_size for f in CACHE_ROOT.rglob("*.npy"))
    print(f"\ncache on disk: {total/1024**3:.2f} GB in "
          f"{sum(1 for _ in CACHE_ROOT.rglob('*.npy'))} files")
else:
    print("\nBUILD_CACHE = False - tensors will be produced from the .npz on the fly.")


class UCFDvsDataset(Dataset):
    # Serves REAL event tensors: [N_SEGMENTS, FRAMES_PER_SEG, 2, HW, HW] float32 in [0,1]
    def __init__(self, items, train=False):
        self.items = items
        self.train = train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        a = self.items[i]
        cp = cache_path_for(a)
        if BUILD_CACHE and cp.exists():
            x = torch.from_numpy(np.load(cp))
            if self.train and AUG_HFLIP and random.random() < 0.5:
                x = torch.flip(x, dims=[-1])
        else:
            x = preprocess_video(a["path"],
                                 jitter=self.train and AUG_SEG_JITTER,
                                 hflip=self.train and AUG_HFLIP and random.random() < 0.5)
        x = x.float() / 255.0
        return x, a["label"], a["cat_idx"], i


train_ds = UCFDvsDataset(train_items, train=True)
val_ds   = UCFDvsDataset(val_items)
test_ds  = UCFDvsDataset(test_items)
print("\ndatasets:", len(train_ds), len(val_ds), len(test_ds))
_x, _lbl, _ci, _ = train_ds[0]
print("sample tensor:", tuple(_x.shape), _x.dtype, "| label", _lbl, "| cat", IDX_TO_CATEGORY[_ci])
del _x

TENSOR CACHE
cache root: G:\Sentrix\training\cache_ucf_crime_dvs\hw128_n32_f4_c32
tag       : hw128_n32_f4_c32

building train cache (1316 videos)...
   {'built': 0, 'reused': 1316, 'failed': 0}

building val cache (233 videos)...
   {'built': 0, 'reused': 233, 'failed': 0}

building test cache (287 videos)...
   {'built': 0, 'reused': 287, 'failed': 0}

cache on disk: 7.17 GB in 1836 files

datasets: 1316 233 287
sample tensor: (32, 4, 2, 128, 128) torch.float32 | label 1 | cat RoadAccidents


---
# CELL 11 — UCF-Crime-DVS model

```text
[B, 32 segments, 4 timesteps, 2 polarities, 128, 128]
                    |
        SpikingEncoder (LIF, rate-coded over the 4 timesteps)
                    |
          segment features  [B, 32, D]
                    |
        MSF-lite: dilated temporal fusion (1 / 2 / 4)
                    |
        +--- anomaly scorer  -> per-segment score  [B, 32]   <-- SENTRIX TCI input
        +--- category head   -> 14 logits (auxiliary, weak)
```

`MSFLite` is *inspired by* the published multi-scale spiking fusion design; it is a compact
re-implementation, not the official one. If you want the published numbers, run the authors'
code — this model is built to slot into SENTRIX, not to chase the leaderboard.

In [23]:
SPIKING = HAS_SPIKINGJELLY
if not SPIKING and not ALLOW_ANN_FALLBACK:
    raise RuntimeError(
        "SpikingJelly is not installed and ALLOW_ANN_FALLBACK is False.\n"
        "   pip install spikingjelly\n"
        "UCF-Crime-DVS is an event/SNN benchmark - the fallback is a deliberate deviation."
    )

LIF_TAU = 2.0
ENC_WIDTHS = (32, 64, 128, 256)
FUSION_HIDDEN = 256
TOPK = 3
DROPOUT = 0.6


def _build_backbone(in_ch, widths, spiking):
    mods, C = [], in_ch
    for w in widths:
        if spiking:
            mods += [
                layer.Conv2d(C, w, 3, stride=1, padding=1, bias=False),
                layer.BatchNorm2d(w),
                neuron.LIFNode(tau=LIF_TAU, surrogate_function=surrogate.ATan(),
                               detach_reset=True),
                layer.MaxPool2d(2),
            ]
        else:
            mods += [
                nn.Conv2d(C, w, 3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(w), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            ]
        C = w
    net = nn.Sequential(*mods)
    if spiking:
        functional.set_step_mode(net, "m")
    return net, C


class SpikingEncoder(nn.Module):
    # [T, B, 2, H, W] -> [B, D]   (rate coding: mean spike rate over the T timesteps)
    def __init__(self, in_ch=2, widths=ENC_WIDTHS, spiking=True):
        super().__init__()
        self.spiking = spiking
        self.net, self.out_dim = _build_backbone(in_ch, widths, spiking)

    def forward(self, x):
        T, B = x.shape[0], x.shape[1]
        if self.spiking:
            y = self.net(x)                              # [T, B, C, h, w]
        else:
            y = self.net(x.reshape(T * B, *x.shape[2:]))
            y = y.reshape(T, B, *y.shape[1:])
        y = y.mean(0)                                    # rate over timesteps
        return F.adaptive_avg_pool2d(y, 1).flatten(1)    # [B, C]


class MSFLite(nn.Module):
    # Multi-scale temporal fusion over MIL segments + anomaly scorer + auxiliary category head.
    def __init__(self, feat_dim, hidden=FUSION_HIDDEN, n_cat=len(UCF_DVS_CATEGORIES)):
        super().__init__()
        self.proj = nn.Linear(feat_dim, hidden)
        self.branches = nn.ModuleList([
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=d, dilation=d) for d in (1, 2, 4)
        ])
        self.fuse = nn.Conv1d(hidden * 3, hidden, kernel_size=1)
        self.norm = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(DROPOUT)
        self.scorer = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(128, 1),
        )
        self.category = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(128, n_cat),
        )

    def forward(self, feats):                 # feats [B, N, feat_dim]
        h = F.relu(self.proj(feats))          # [B, N, H]
        z = h.transpose(1, 2)                 # [B, H, N]
        multi = torch.cat([b(z) for b in self.branches], dim=1)
        z = F.relu(self.fuse(multi)) + z
        h = self.norm(z.transpose(1, 2))      # [B, N, H]
        h = self.drop(h)
        seg_logits = self.scorer(h).squeeze(-1)   # [B, N]
        return seg_logits, h


class UCFDvsAnomalyNet(nn.Module):
    def __init__(self, spiking=True):
        super().__init__()
        self.encoder = SpikingEncoder(2, ENC_WIDTHS, spiking)
        self.fusion = MSFLite(self.encoder.out_dim)
        self.spiking = spiking

    def forward(self, x):                     # x [B, N, F, 2, H, W]
        B, N, Fr, C, H, W = x.shape
        z = x.permute(2, 0, 1, 3, 4, 5).reshape(Fr, B * N, C, H, W)
        if self.spiking:
            functional.reset_net(self.encoder)
        feats = self.encoder(z).view(B, N, -1)
        seg_logits, h = self.fusion(feats)
        return seg_logits, h

    def category_from_topk(self, h, seg_logits, k=TOPK):
        idx = seg_logits.topk(min(k, seg_logits.shape[1]), dim=1).indices      # [B,k]
        pooled = torch.gather(h, 1, idx.unsqueeze(-1).expand(-1, -1, h.shape[-1])).mean(1)
        return self.category(pooled) if hasattr(self, "category") else self.fusion.category(pooled)


set_seed()
model = UCFDvsAnomalyNet(spiking=SPIKING).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())

print("=" * 64)
print("MODEL")
print("=" * 64)
print("backend        :", "SpikingJelly LIF (spiking)" if SPIKING else "ANN FALLBACK (deviation)")
print("encoder widths :", ENC_WIDTHS, "-> feature dim", model.encoder.out_dim)
print("segments (MIL) :", N_SEGMENTS, "| timesteps:", FRAMES_PER_SEG)
print("parameters     :", f"{n_params/1e6:.2f} M")

with torch.no_grad():
    _x, _l, _c, _ = train_ds[0]
    _s, _h = model(_x.unsqueeze(0).to(DEVICE))
    print("forward check  : seg_logits", tuple(_s.shape), "| features", tuple(_h.shape))
    del _x, _s, _h
if DEVICE == "cuda":
    torch.cuda.empty_cache()

MODEL
backend        : SpikingJelly LIF (spiking)
encoder widths : (32, 64, 128, 256) -> feature dim 256
segments (MIL) : 32 | timesteps: 4
parameters     : 1.31 M
forward check  : seg_logits (1, 32) | features (1, 32, 256)


---
# CELL 12 — Training (weakly-supervised MIL)

The supervision available is **video-level**: a video from `Robbery/` contains a robbery
*somewhere*, and a video from `Normal_Videos/` contains none. That is exactly the setting
multiple-instance learning is for, and it is why this notebook does not pretend to have
frame-level labels.

Loss terms:

| Term | Purpose |
|---|---|
| ranking hinge (top-k) | the worst segment of an anomalous video must outscore the worst of a normal one |
| video-level BCE | stabilises early epochs |
| smoothness | adjacent segment scores should not oscillate |
| sparsity | anomalies are rare inside a video |
| auxiliary CE | 14-way category, on the top-k segments, **weak labels — context only** |

In [24]:
from torch.utils.checkpoint import checkpoint

ENC_CHUNK      = 64      # sequences encoded per chunk; halve if still tight
USE_CHECKPOINT = True    # recompute encoder activations in backward


class UCFDvsAnomalyNet(nn.Module):
    def __init__(self, spiking=True):
        super().__init__()
        self.encoder = SpikingEncoder(2, ENC_WIDTHS, spiking)
        self.fusion = MSFLite(self.encoder.out_dim)
        self.spiking = spiking

    def _encode(self, zc):
        # reset INSIDE the checkpointed function so the backward recomputation
        # starts from the same membrane state as the forward pass
        if self.spiking:
            functional.reset_net(self.encoder)
        return self.encoder(zc)

    def forward(self, x):                      # x [B, N, F, 2, H, W]
        B, N, Fr, C, H, W = x.shape
        z = x.permute(2, 0, 1, 3, 4, 5).reshape(Fr, B * N, C, H, W)
        outs = []
        for i in range(0, B * N, ENC_CHUNK):
            zc = z[:, i:i + ENC_CHUNK]
            if USE_CHECKPOINT and self.training and torch.is_grad_enabled():
                outs.append(checkpoint(self._encode, zc, use_reentrant=False))
            else:
                outs.append(self._encode(zc))
        feats = torch.cat(outs, dim=0).view(B, N, -1)
        seg_logits, h = self.fusion(feats)
        return seg_logits, h

    def category_from_topk(self, h, seg_logits, k=TOPK):
        idx = seg_logits.topk(min(k, seg_logits.shape[1]), dim=1).indices
        pooled = torch.gather(h, 1, idx.unsqueeze(-1).expand(-1, -1, h.shape[-1])).mean(1)
        return self.fusion.category(pooled)


set_seed()
del model
import gc; gc.collect(); torch.cuda.empty_cache()
model = UCFDvsAnomalyNet(spiking=SPIKING).to(DEVICE)

print("=" * 64)
print("MODEL REBUILT (chunked + gradient-checkpointed encoder)")
print("=" * 64)
print("chunk size     :", ENC_CHUNK, "| checkpointing:", USE_CHECKPOINT)
print("parameters     :", f"{sum(p.numel() for p in model.parameters())/1e6:.2f} M")

# ---- VRAM fit test: one real forward+backward at the training batch size ----
PROBE_VIDEOS = 4          # must equal 2 * BATCH_VIDEOS from CELL 12
torch.cuda.reset_peak_memory_stats()
model.train()
_xb = torch.stack([train_ds[i][0] for i in range(PROBE_VIDEOS)]).to(DEVICE)
_opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
with (torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()):
    _sl, _h = model(_xb)
    _loss = torch.sigmoid(_sl.float()).mean() + _h.float().mean()
_loss.backward()
_opt.zero_grad(set_to_none=True)
peak = torch.cuda.max_memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\npeak VRAM for {PROBE_VIDEOS} videos: {peak:.2f} GB / {total:.1f} GB")
print("OK - proceed to CELL 12" if peak < total * 0.75 else
      "TIGHT - halve ENC_CHUNK (or set BATCH_VIDEOS=1) and rerun this cell")
del _xb, _sl, _h, _loss, _opt
gc.collect(); torch.cuda.empty_cache()

MODEL REBUILT (chunked + gradient-checkpointed encoder)
chunk size     : 64 | checkpointing: True
parameters     : 1.31 M

peak VRAM for 4 videos: 5.53 GB / 8.0 GB
OK - proceed to CELL 12


In [25]:
EPOCHS          = 30
BATCH_VIDEOS    = 2        # anomalous per step; the same number of normals is paired with them
LR              = 1e-4
WEIGHT_DECAY    = 5e-4
LAMBDA_SMOOTH   = 8e-4
LAMBDA_SPARSE   = 8e-4
LAMBDA_BCE      = 0.5
AUX_WEIGHT      = 0.3      # auxiliary category head - never wired into TCI
PATIENCE        = 8
GRAD_CLIP       = 5.0

from torch.utils.data import Subset
from sklearn.metrics import roc_auc_score, average_precision_score

anom_idx = [i for i, a in enumerate(train_items) if a["label"] == 1]
norm_idx = [i for i, a in enumerate(train_items) if a["label"] == 0]
if not anom_idx or not norm_idx:
    raise RuntimeError("MIL needs both anomalous and normal videos in the training split.")

anom_loader = DataLoader(Subset(train_ds, anom_idx), batch_size=BATCH_VIDEOS, shuffle=True,
                         num_workers=NUM_WORKERS, drop_last=True,
                         pin_memory=(DEVICE == "cuda"))
norm_loader = DataLoader(Subset(train_ds, norm_idx), batch_size=BATCH_VIDEOS, shuffle=True,
                         num_workers=NUM_WORKERS, drop_last=True,
                         pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

print("MIL loaders: anomalous batches", len(anom_loader), "| normal batches", len(norm_loader))

from contextlib import nullcontext
def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def video_scores(seg_logits, k=TOPK):
    s = torch.sigmoid(seg_logits)
    return s.topk(min(k, s.shape[1]), dim=1).values.mean(1), s


def mil_objective(seg_logits, labels, h, cat_idx, net):
    vs, s = video_scores(seg_logits)
    labels = labels.float()
    anom, norm = vs[labels == 1], vs[labels == 0]

    if anom.numel() and norm.numel():
        ranking = F.relu(1.0 - anom.unsqueeze(1) + norm.unsqueeze(0)).mean()
    else:
        ranking = seg_logits.sum() * 0.0

    _no_amp = torch.autocast(device_type="cuda", enabled=False) if USE_AMP else nullcontext()
    with _no_amp:
        bce = F.binary_cross_entropy(vs.float().clamp(1e-6, 1 - 1e-6), labels.float())
    smooth = ((s[:, 1:] - s[:, :-1]) ** 2).sum(dim=1).mean()
    sparse = s[labels == 1].sum(dim=1).mean() if (labels == 1).any() else s.sum() * 0.0

    cat_logits = net.fusion.category(
        torch.gather(h, 1, seg_logits.topk(min(TOPK, seg_logits.shape[1]), dim=1)
                     .indices.unsqueeze(-1).expand(-1, -1, h.shape[-1])).mean(1))
    aux = F.cross_entropy(cat_logits, cat_idx)

    total = (ranking + LAMBDA_BCE * bce
             + LAMBDA_SMOOTH * smooth + LAMBDA_SPARSE * sparse
             + AUX_WEIGHT * aux)
    return total, {"rank": float(ranking), "bce": float(bce), "smooth": float(smooth),
                   "sparse": float(sparse), "aux": float(aux)}


@torch.no_grad()
def evaluate(loader, net):
    net.eval()
    vids, ys, cats, scores, cat_pred, cat_conf, seg_store = [], [], [], [], [], [], {}
    for x, y, ci, idx in loader:
        x = x.to(DEVICE, non_blocking=True)
        with amp_ctx():
            seg_logits, h = net(x)
        vs, s = video_scores(seg_logits.float())
        cl = net.fusion.category(
            torch.gather(h.float(), 1,
                         seg_logits.float().topk(min(TOPK, seg_logits.shape[1]), dim=1)
                         .indices.unsqueeze(-1).expand(-1, -1, h.shape[-1])).mean(1))
        prob = torch.softmax(cl, dim=1)
        conf, pred = prob.max(1)
        for b in range(x.shape[0]):
            j = int(idx[b])
            item = loader.dataset.items[j] if hasattr(loader.dataset, "items") \
                else loader.dataset.dataset.items[j]
            vids.append(item["stem"])
            seg_store[item["stem"]] = s[b].detach().cpu().numpy()
            cat_pred.append(int(pred[b])); cat_conf.append(float(conf[b]))
        ys += [int(v) for v in y]
        cats += [int(v) for v in ci]
        scores += [float(v) for v in vs]
    return {"video_id": vids, "y": np.array(ys), "cat": np.array(cats),
            "score": np.array(scores), "cat_pred": np.array(cat_pred),
            "cat_conf": np.array(cat_conf), "segments": seg_store}


def auc_ap(res):
    y, s = res["y"], res["score"]
    if len(set(y.tolist())) < 2:
        return float("nan"), float("nan")
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s))


set_seed()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = make_scaler()

CKPT = UCF_DVS_RUN_DIR / "best_checkpoint.pt"
history, best_auc, best_epoch, bad = [], -1.0, -1, 0
steps_per_epoch = min(len(anom_loader), len(norm_loader))

print("\n" + "=" * 64)
print("TRAINING (weakly-supervised MIL)")
print("=" * 64)
print(f"videos: train {len(train_items)} | val {len(val_items)} | test {len(test_items)}")
print(f"epochs {EPOCHS} | steps/epoch {steps_per_epoch} | pairs/step {BATCH_VIDEOS}+{BATCH_VIDEOS}")

t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    it_a, it_n = iter(anom_loader), iter(norm_loader)
    agg, t0 = {}, time.time()
    for step in range(steps_per_epoch):
        xa, ya, ca, _ = next(it_a)
        xn, yn, cn, _ = next(it_n)
        x = torch.cat([xa, xn]).to(DEVICE, non_blocking=True)
        y = torch.cat([ya, yn]).to(DEVICE)
        ci = torch.cat([ca, cn]).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with amp_ctx():
            seg_logits, h = model(x)
        # Loss OUTSIDE autocast: binary_cross_entropy is on the CUDA autocast
        # banlist, and fp32 loss math is the recommended pattern regardless.
        loss, parts = mil_objective(seg_logits.float(), y, h.float(), ci, model)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        for k, v in parts.items():
            agg[k] = agg.get(k, 0.0) + v
        agg["loss"] = agg.get("loss", 0.0) + float(loss)

    scheduler.step()
    torch.cuda.empty_cache() 
    res = evaluate(val_loader, model)
    v_auc, v_ap = auc_ap(res)
    row = {"epoch": epoch, **{k: v / steps_per_epoch for k, v in agg.items()},
           "val_auc": v_auc, "val_ap": v_ap, "seconds": time.time() - t0}
    history.append(row)

    flag = ""
    if not math.isnan(v_auc) and v_auc > best_auc + 1e-5:
        best_auc, best_epoch, bad = v_auc, epoch, 0
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch,
                    "val_auc": v_auc, "spiking": SPIKING}, CKPT)
        flag = "  <-- best (checkpoint saved)"
    else:
        bad += 1
    print(f"epoch {epoch:>3}/{EPOCHS} | loss {row['loss']:.4f} (rank {row['rank']:.3f} "
          f"bce {row['bce']:.3f} aux {row['aux']:.3f}) | val AUC {v_auc:.4f} "
          f"AP {v_ap:.4f} | {row['seconds']:.1f}s{flag}")
    if bad >= PATIENCE:
        print(f"early stopping at epoch {epoch}")
        break

TRAIN_SECONDS = time.time() - t_start
import pandas as pd
pd.DataFrame(history).to_csv(UCF_DVS_RUN_DIR / "history.csv", index=False)
print(f"\ntraining time: {TRAIN_SECONDS/60:.1f} min | best val AUC {best_auc:.4f} @ epoch {best_epoch}")

MIL loaders: anomalous batches 340 | normal batches 318

TRAINING (weakly-supervised MIL)
videos: train 1316 | val 233 | test 287
epochs 30 | steps/epoch 318 | pairs/step 2+2
epoch   1/30 | loss 1.9630 (rank 0.995 bce 0.695 aux 2.041) | val AUC 0.5527 AP 0.5169 | 261.5s  <-- best (checkpoint saved)
epoch   2/30 | loss 1.9409 (rank 0.997 bce 0.695 aux 1.964) | val AUC 0.6058 AP 0.6114 | 241.4s  <-- best (checkpoint saved)
epoch   3/30 | loss 1.9429 (rank 0.999 bce 0.698 aux 1.958) | val AUC 0.6421 AP 0.6487 | 240.9s  <-- best (checkpoint saved)
epoch   4/30 | loss 1.9331 (rank 0.998 bce 0.697 aux 1.931) | val AUC 0.6094 AP 0.6356 | 240.9s
epoch   5/30 | loss 1.9334 (rank 0.998 bce 0.698 aux 1.929) | val AUC 0.6291 AP 0.7115 | 240.9s
epoch   6/30 | loss 1.8389 (rank 0.936 bce 0.666 aux 1.876) | val AUC 0.7029 AP 0.7424 | 240.9s  <-- best (checkpoint saved)
epoch   7/30 | loss 1.7183 (rank 0.835 bce 0.659 aux 1.821) | val AUC 0.7498 AP 0.7899 | 240.8s  <-- best (checkpoint saved)
epoch   

---
# CELL 13 — Validation

Reloads the best checkpoint and reports the validation numbers that model selection was
based on. The test split has not been touched at this point.

In [26]:
ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model_state_dict"])
print("loaded best checkpoint from epoch", ck["epoch"])

val_res = evaluate(val_loader, model)
val_auc, val_ap = auc_ap(val_res)

print("\n" + "=" * 64)
print("VALIDATION (video-level, real labels)")
print("=" * 64)
print(f"videos       : {len(val_res['y'])}  (anomalous {int(val_res['y'].sum())})")
print(f"ROC-AUC      : {val_auc:.4f}")
print(f"AP (PR-AUC)  : {val_ap:.4f}")
print(f"score mean   : normal {val_res['score'][val_res['y']==0].mean():.4f} | "
      f"anomalous {val_res['score'][val_res['y']==1].mean():.4f}")

# operating threshold chosen on VALIDATION only
from sklearn.metrics import precision_recall_curve
prec, rec, thr = precision_recall_curve(val_res["y"], val_res["score"])
f1s = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)
best_i = int(np.nanargmax(f1s[:-1])) if len(thr) else 0
ANOMALY_THRESHOLD = float(thr[best_i]) if len(thr) else 0.5
print(f"\noperating threshold (max F1 on val): {ANOMALY_THRESHOLD:.4f}  "
      f"(F1 {f1s[best_i]:.4f})")
idx95 = np.where(rec[:-1] >= 0.95)[0]
THRESHOLD_95_RECALL = float(thr[idx95[-1]]) if len(idx95) and len(thr) else None
print("threshold at 95% recall           :",
      f"{THRESHOLD_95_RECALL:.4f}" if THRESHOLD_95_RECALL is not None else "not reachable")

loaded best checkpoint from epoch 10

VALIDATION (video-level, real labels)
videos       : 233  (anomalous 120)
ROC-AUC      : 0.8263
AP (PR-AUC)  : 0.8587
score mean   : normal 0.2450 | anomalous 0.6779

operating threshold (max F1 on val): 0.1507  (F1 0.7615)
threshold at 95% recall           : 0.0127


---
# CELL 14 — Test evaluation

Video-level AUC on the official test split, plus **frame-level AUC** when the official
`gt-ucf.npy` / `gt-ucf-dic.pickle` is available. If it is not, the notebook says so instead of
manufacturing frame labels.

In [27]:
test_res = evaluate(test_loader, model)
test_auc, test_ap = auc_ap(test_res)

print("=" * 64)
print("TEST — VIDEO LEVEL")
print("=" * 64)
print("split source :", SPLIT_SOURCE)
print(f"videos       : {len(test_res['y'])}  (anomalous {int(test_res['y'].sum())})")
print(f"ROC-AUC      : {test_auc:.4f}")
print(f"AP (PR-AUC)  : {test_ap:.4f}")

pred_bin = (test_res["score"] >= ANOMALY_THRESHOLD).astype(int)
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
p, r, f1, _ = precision_recall_fscore_support(test_res["y"], pred_bin,
                                              average="binary", zero_division=0)
tn, fp, fn, tp = confusion_matrix(test_res["y"], pred_bin, labels=[0, 1]).ravel()
print(f"\nat threshold {ANOMALY_THRESHOLD:.3f}: P {p:.4f}  R {r:.4f}  F1 {f1:.4f}")
print(f"TP {tp}  FP {fp}  FN {fn}  TN {tn}   FNR {fn/max(fn+tp,1):.4f}")

# ---------------- frame-level evaluation ----------------
FRAME_AUC = None
FRAME_GT_SOURCE = None

def _looks_like_frame_mask(g, n_segments=N_SEGMENTS):
    g = np.asarray(g).ravel()
    if g.size < max(8 * n_segments, 100):
        return False, f"length {g.size} is too short to be a frame timeline"
    u = np.unique(g)
    if u.size > 2 or not np.isin(u, [0, 1]).all():
        return False, f"values are not 0/1 (sample {u[:6].tolist()})"
    return True, "ok"


def _validate_gt(mapping, source):
    sample = list(mapping.items())[:20]
    checks = [(k,) + _looks_like_frame_mask(v) for k, v in sample]
    ok = sum(1 for c in checks if c[1])
    if ok >= max(1, int(0.8 * len(checks))):
        return mapping, source
    print("\n" + "=" * 64)
    print("FRAME-LEVEL GT REJECTED")
    print("=" * 64)
    print(f"{source}: only {ok}/{len(checks)} entries look like per-frame 0/1 masks.")
    for k, good, why in checks[:5]:
        print(f"   {k}: {why}")
    print("This file is probably temporal annotations (start/end frame pairs), not a")
    print("per-frame mask. Expanding it would produce a meaningless AUC, so it is not used.")
    return None, None


def load_frame_gt():
    if UCF_DVS_GT_PICKLE.exists():
        try:
            with open(UCF_DVS_GT_PICKLE, "rb") as fh:
                obj = pickle.load(fh)
            if isinstance(obj, dict):
                return _validate_gt({norm_stem(k): np.asarray(v).ravel()
                                     for k, v in obj.items()}, UCF_DVS_GT_PICKLE.name)
        except Exception as e:
            print("gt pickle could not be read:", e)
    if UCF_DVS_GT_NPY.exists():
        try:
            arr = np.load(UCF_DVS_GT_NPY, allow_pickle=True)
            if isinstance(arr, np.ndarray) and arr.dtype == object and arr.shape == ():
                obj = arr.item()
                if isinstance(obj, dict):
                    return _validate_gt({norm_stem(k): np.asarray(v).ravel()
                                         for k, v in obj.items()}, UCF_DVS_GT_NPY.name)
            print("gt-ucf.npy is a flat array of length", np.asarray(arr).ravel().shape[0],
                  "- a per-video mapping is required to align it, so it is not used.")
        except Exception as e:
            print("gt npy could not be read:", e)
    return None, None

TEST — VIDEO LEVEL
split source : official train_split.txt / test_split.txt
videos       : 287  (anomalous 140)
ROC-AUC      : 0.1940
AP (PR-AUC)  : 0.3372

at threshold 0.151: P 0.4538  R 0.8429  F1 0.5900
TP 118  FP 142  FN 22  TN 5   FNR 0.1571


---
# CELL 15 — Confusion matrix / classification metrics

Two separate reports, deliberately not merged:

1. **binary anomaly** — the number that matters for SENTRIX
2. **14-way category** — auxiliary, weakly supervised, shown for operator context only

In [28]:
from sklearn.metrics import classification_report

print("=" * 70)
print("1. BINARY ANOMALY (SENTRIX-relevant)")
print("=" * 70)
print(classification_report(test_res["y"], pred_bin,
                            target_names=["normal", "anomalous"], zero_division=0))
cm2 = confusion_matrix(test_res["y"], pred_bin, labels=[0, 1])
print("confusion matrix (rows=true, cols=pred):")
print(f"{'':12}{'normal':>12}{'anomalous':>12}")
for i, nm in enumerate(["normal", "anomalous"]):
    print(f"{nm:>11} " + "".join(f"{v:>12}" for v in cm2[i]))

print("\n" + "=" * 70)
print("2. CATEGORY — AUXILIARY, WEAKLY SUPERVISED (do NOT wire into TCI)")
print("=" * 70)
present = sorted(set(test_res["cat"].tolist()) | set(test_res["cat_pred"].tolist()))
names = [IDX_TO_CATEGORY[i] for i in present]
print(classification_report(test_res["cat"], test_res["cat_pred"],
                            labels=present, target_names=names, zero_division=0))
cm14 = confusion_matrix(test_res["cat"], test_res["cat_pred"], labels=present)
print("confusion matrix (rows=true, cols=pred):")
hdr = " " * 16 + "".join(f"{n[:9]:>10}" for n in names)
print(hdr)
for i, n in enumerate(names):
    print(f"{n[:15]:>15} " + "".join(f"{v:>10}" for v in cm14[i]))

cat_acc = float((test_res["cat"] == test_res["cat_pred"]).mean())
print(f"\ncategory accuracy: {cat_acc:.4f}")
print("Reminder: these labels are propagated from video level to the top-k segments.")
print("They are context for an operator, not frame-level ground truth.")

pd.DataFrame(cm2, index=["normal", "anomalous"],
             columns=["normal", "anomalous"]).to_csv(
    UCF_DVS_RUN_DIR / "confusion_binary.csv")
pd.DataFrame(cm14, index=names, columns=names).to_csv(
    UCF_DVS_RUN_DIR / "confusion_category.csv")
print("\nsaved confusion matrices to", UCF_DVS_RUN_DIR)

1. BINARY ANOMALY (SENTRIX-relevant)
              precision    recall  f1-score   support

      normal       0.19      0.03      0.06       147
   anomalous       0.45      0.84      0.59       140

    accuracy                           0.43       287
   macro avg       0.32      0.44      0.32       287
weighted avg       0.32      0.43      0.32       287

confusion matrix (rows=true, cols=pred):
                  normal   anomalous
     normal            5         142
  anomalous           22         118

2. CATEGORY — AUXILIARY, WEAKLY SUPERVISED (do NOT wire into TCI)
               precision    recall  f1-score   support

        Abuse       0.00      0.00      0.00         2
       Arrest       0.00      0.00      0.00         5
        Arson       0.00      0.00      0.00         9
      Assault       0.00      0.00      0.00         3
     Burglary       0.06      0.92      0.11        13
    Explosion       0.00      0.00      0.00        21
     Fighting       0.00      0

---
# CELL 16 — Save model

In [29]:
torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": "UCFDvsAnomalyNet (SpikingEncoder + MSFLite)",
    "spiking_backend": bool(SPIKING),
    "encoder_widths": list(ENC_WIDTHS),
    "feature_dim": model.encoder.out_dim,
    "fusion_hidden": FUSION_HIDDEN,
    "lif_tau": LIF_TAU,
    "categories": UCF_DVS_CATEGORIES,
    "normal_category": NORMAL_CATEGORY,
    "n_segments": N_SEGMENTS,
    "frames_per_segment": FRAMES_PER_SEG,
    "target_hw": TARGET_HW,
    "event_clip": EVENT_CLIP,
    "topk": TOPK,
    "anomaly_threshold": ANOMALY_THRESHOLD,
    "threshold_95_recall": THRESHOLD_95_RECALL,
    "val_auc": val_auc,
    "test_auc": test_auc,
    "frame_auc": FRAME_AUC,
    "split_source": SPLIT_SOURCE,
    "run_id": RUN_ID,
    "training_type": "real_data",
    "synthetic_data": False,
}, UCF_DVS_MODEL)

print("=" * 64)
print("MODEL SAVED")
print("=" * 64)
print(UCF_DVS_MODEL, f"({UCF_DVS_MODEL.stat().st_size/1024**2:.2f} MB)")
print("\nNothing outside G:\\Sentrix was written. The dataset root is untouched.")

MODEL SAVED
G:\Sentrix\backend\models\v2_real\ucf_crime_dvs\ucf_crime_dvs_anomaly_v1.pt (5.02 MB)

Nothing outside G:\Sentrix was written. The dataset root is untouched.


---
# CELL 17 — Save preprocessing + class metadata

The SENTRIX runtime must reproduce this preprocessing exactly, or the scores will not mean
what they meant at training time.

In [30]:
metadata = {
    "model_name": UCF_DVS_MODEL.stem,
    "created": datetime.now().isoformat(timespec="seconds"),
    "run_id": RUN_ID,
    "architecture": "SpikingEncoder (LIF) + MSFLite multi-scale temporal fusion",
    "spiking_backend": bool(SPIKING),
    "task": "weakly-supervised video anomaly detection (event / DVS)",
    "dataset": str(UCF_DVS_ROOT),
    "dataset_copied": False,
    "split_source": SPLIT_SOURCE,
    "train_videos": len(train_items),
    "val_videos": len(val_items),
    "test_videos": len(test_items),

    "preprocessing": {
        "input_format": "UCF-Crime-DVS .npz event data",
        "npz_layout": NPZ_LAYOUT,
        "frame_key": FRAME_KEY,
        "native_hw": [EVENT_H, EVENT_W],
        "polarity_channels": 2,
        "spatial_resize": [TARGET_HW, TARGET_HW],
        "spatial_method": "adaptive MAX pool (sparse-event friendly)",
        "n_segments": N_SEGMENTS,
        "frames_per_segment": FRAMES_PER_SEG,
        "temporal_sampling": "evenly spaced real frames within each segment",
        "event_clip": EVENT_CLIP,
        "value_scaling": "clip(counts, 0, EVENT_CLIP) / EVENT_CLIP -> [0,1]",
        "tensor_shape": [N_SEGMENTS, FRAMES_PER_SEG, 2, TARGET_HW, TARGET_HW],
        "augmentation_train_only": ["horizontal_flip", "segment_jitter"],
    },

    "classes": {
        "binary": {"normal": 0, "anomalous": 1},
        "categories": UCF_DVS_CATEGORIES,
        "category_index": CATEGORY_TO_IDX,
        "normal_category": NORMAL_CATEGORY,
    },

    "scoring": {
        "video_score": f"mean of the top-{TOPK} segment sigmoid scores",
        "anomaly_threshold": ANOMALY_THRESHOLD,
        "threshold_95_recall": THRESHOLD_95_RECALL,
        "threshold_selected_on": "validation split only",
    },

    "sentrix_contract": {
        "consumed_by_tci": ["anomaly_score"],
        "context_only": ["predicted_category", "confidence"],
        "note": "The 14-way category head is trained on weak video-level labels propagated "
                "to top-k segments. It must NOT be treated as frame-level ground truth and "
                "must NOT be wired directly into the TCI fusion model.",
    },

    "metrics": {
        "val_auc": val_auc, "val_ap": val_ap,
        "test_auc": test_auc, "test_ap": test_ap,
        "test_binary_precision": float(p), "test_binary_recall": float(r),
        "test_binary_f1": float(f1),
        "test_false_negative_rate": float(fn / max(fn + tp, 1)),
        "frame_auc": FRAME_AUC,
        "frame_gt_source": FRAME_GT_SOURCE,
        "category_accuracy_weak": cat_acc,
    },

    "training": {
        "epochs_run": len(history), "best_epoch": best_epoch,
        "lr": LR, "weight_decay": WEIGHT_DECAY, "batch_videos": BATCH_VIDEOS,
        "loss": "MIL ranking + video BCE + smoothness + sparsity + auxiliary CE",
        "lambda_smooth": LAMBDA_SMOOTH, "lambda_sparse": LAMBDA_SPARSE,
        "aux_weight": AUX_WEIGHT, "seed": SEED, "device": DEVICE,
        "training_minutes": round(TRAIN_SECONDS / 60, 1),
    },

    "training_type": "real_data",
    "synthetic_data": False,
}

UCF_DVS_METADATA.write_text(json.dumps(metadata, indent=4, default=str), encoding="utf-8")
print("metadata written:", UCF_DVS_METADATA)
print(json.dumps({k: metadata[k] for k in
                  ["architecture", "split_source", "scoring", "sentrix_contract"]},
                 indent=2, default=str))

metadata written: G:\Sentrix\backend\models\v2_real\ucf_crime_dvs\ucf_crime_dvs_anomaly_v1_metadata.json
{
  "architecture": "SpikingEncoder (LIF) + MSFLite multi-scale temporal fusion",
  "split_source": "official train_split.txt / test_split.txt",
  "scoring": {
    "video_score": "mean of the top-3 segment sigmoid scores",
    "anomaly_threshold": 0.1507003903388977,
    "threshold_95_recall": 0.012715784832835197,
    "threshold_selected_on": "validation split only"
  },
  "sentrix_contract": {
    "consumed_by_tci": [
      "anomaly_score"
    ],
    "context_only": [
      "predicted_category",
      "confidence"
    ],
    "note": "The 14-way category head is trained on weak video-level labels propagated to top-k segments. It must NOT be treated as frame-level ground truth and must NOT be wired directly into the TCI fusion model."
  }
}


---
# CELL 18 — Save results + SENTRIX inference API

Produces the record the fusion layer consumes:

```text
video_id, clip_id, anomaly_score, predicted_category, confidence
```

`anomaly_score` is the only field the TCI model should ever see.

In [31]:
rows = []
for i, stem in enumerate(test_res["video_id"]):
    seg = test_res["segments"][stem]
    for clip_id, sc in enumerate(seg):
        rows.append({
            "video_id": stem,
            "clip_id": clip_id,
            "anomaly_score": float(sc),
            "predicted_category": IDX_TO_CATEGORY[int(test_res["cat_pred"][i])],
            "confidence": float(test_res["cat_conf"][i]),
            "video_anomaly_score": float(test_res["score"][i]),
            "true_category": IDX_TO_CATEGORY[int(test_res["cat"][i])],
            "true_binary": int(test_res["y"][i]),
        })

preds = pd.DataFrame(rows)
preds.to_csv(UCF_DVS_PREDS_CSV, index=False)
print("clip-level predictions:", UCF_DVS_PREDS_CSV, f"({len(preds)} rows)")

results = {
    "run_id": RUN_ID,
    "split_source": SPLIT_SOURCE,
    "val": {"auc": val_auc, "ap": val_ap, "n": int(len(val_res["y"]))},
    "test": {"auc": test_auc, "ap": test_ap, "n": int(len(test_res["y"])),
             "precision": float(p), "recall": float(r), "f1": float(f1),
             "false_negative_rate": float(fn / max(fn + tp, 1)),
             "frame_auc": FRAME_AUC, "frame_gt_source": FRAME_GT_SOURCE},
    "category_accuracy_weak": cat_acc,
    "anomaly_threshold": ANOMALY_THRESHOLD,
    "model": str(UCF_DVS_MODEL),
    "metadata": str(UCF_DVS_METADATA),
    "predictions_csv": str(UCF_DVS_PREDS_CSV),
}
(UCF_DVS_RUN_DIR / f"results_{RUN_ID}.json").write_text(
    json.dumps(results, indent=2, default=str), encoding="utf-8")

report = [
    "```text",
    "=" * 66,
    "SENTRIX — UCF-CRIME-DVS EVENT ANOMALY MODEL",
    "=" * 66,
    f"run id        : {RUN_ID}",
    f"dataset       : {UCF_DVS_ROOT}",
    f"split source  : {SPLIT_SOURCE}",
    f"backend       : {'spiking (SpikingJelly LIF)' if SPIKING else 'ANN FALLBACK - deviation'}",
    "",
    f"videos        : train {len(train_items)} | val {len(val_items)} | test {len(test_items)}",
    "",
    f"val   ROC-AUC : {val_auc:.4f}   AP {val_ap:.4f}",
    f"test  ROC-AUC : {test_auc:.4f}   AP {test_ap:.4f}",
    f"test  P/R/F1  : {p:.4f} / {r:.4f} / {f1:.4f}  @ threshold {ANOMALY_THRESHOLD:.3f}",
    f"test  FNR     : {fn/max(fn+tp,1):.4f}",
    f"frame ROC-AUC : {FRAME_AUC if FRAME_AUC is not None else 'not evaluated (no official GT)'}",
    f"category acc  : {cat_acc:.4f}  (weak labels - context only)",
    "",
    "SENTRIX contract: TCI consumes anomaly_score ONLY.",
    "=" * 66,
    "```",
]
(UCF_DVS_RUN_DIR / "EVALUATION_REPORT.md").write_text("\n".join(report), encoding="utf-8")
print("\n".join(report))


# ---------------------------------------------------------------
# SENTRIX-facing inference API
# ---------------------------------------------------------------
@torch.no_grad()
def sentrix_dvs_infer(npz_path, net=None, threshold=None):
    # REAL .npz in -> SENTRIX record out. Reproduces the training preprocessing exactly.
    net = net or model
    threshold = ANOMALY_THRESHOLD if threshold is None else threshold
    net.eval()
    path = Path(npz_path)
    x = preprocess_video(path).float().div(255.0).unsqueeze(0).to(DEVICE)
    seg_logits, h = net(x)
    vs, s = video_scores(seg_logits.float())
    cl = net.fusion.category(
        torch.gather(h.float(), 1,
                     seg_logits.float().topk(min(TOPK, seg_logits.shape[1]), dim=1)
                     .indices.unsqueeze(-1).expand(-1, -1, h.shape[-1])).mean(1))
    prob = torch.softmax(cl, dim=1)[0]
    conf, pred = float(prob.max()), int(prob.argmax())
    seg = s[0].detach().cpu().numpy()
    return {
        "video_id": path.stem,
        "anomaly_score": float(vs[0]),          # <-- the ONLY field TCI should consume
        "is_anomalous": bool(float(vs[0]) >= threshold),
        "threshold": float(threshold),
        "predicted_category": IDX_TO_CATEGORY[pred],   # context only
        "confidence": conf,                            # context only
        "clips": [{"clip_id": i, "anomaly_score": float(v)} for i, v in enumerate(seg)],
    }


print("\n" + "=" * 66)
print("SENTRIX INFERENCE API — example on a real test file")
print("=" * 66)
_demo = sentrix_dvs_infer(test_items[0]["path"])
print(json.dumps({k: v for k, v in _demo.items() if k != "clips"}, indent=2))
print(f"clips: {len(_demo['clips'])}  (first 3: "
      f"{[round(c['anomaly_score'],4) for c in _demo['clips'][:3]]})")

print("\n".join([
    "",
    "-" * 68,
    "HOW NOTEBOOK 2 (V3) CONSUMES THIS",
    "-" * 68,
    "The TCI fusion module expects an `anomaly_score` column per event.",
    "Feed it from here:",
    "",
    "    rec = sentrix_dvs_infer(clip_npz_path)",
    "    fusion_row['anomaly_score'] = rec['anomaly_score']",
    "",
    "Do NOT feed `predicted_category` or the 14-way logits into the fusion",
    "model. They are weakly supervised context for the operator UI.",
    "-" * 68,
]))

clip-level predictions: G:\Sentrix\training\runs_ucf_crime_dvs\sentrix_dvs_predictions.csv (9184 rows)
```text
SENTRIX — UCF-CRIME-DVS EVENT ANOMALY MODEL
run id        : 20260824_132146
dataset       : G:\event_frame_duration533326\event_frame_duration533326
split source  : official train_split.txt / test_split.txt
backend       : spiking (SpikingJelly LIF)

videos        : train 1316 | val 233 | test 287

val   ROC-AUC : 0.8263   AP 0.8587
test  ROC-AUC : 0.1940   AP 0.3372
test  P/R/F1  : 0.4538 / 0.8429 / 0.5900  @ threshold 0.151
test  FNR     : 0.1571
frame ROC-AUC : not evaluated (no official GT)
category acc  : 0.0767  (weak labels - context only)

SENTRIX contract: TCI consumes anomaly_score ONLY.
```

SENTRIX INFERENCE API — example on a real test file
{
  "video_id": "Abuse028_x264_87",
  "anomaly_score": 0.9503854513168335,
  "is_anomalous": true,
  "threshold": 0.1507003903388977,
  "predicted_category": "Burglary",
  "confidence": 0.20456111431121826
}
clips: 32  (first 3

---
---
# APPENDIX — Conda environment for the event/SNN notebook

This notebook needs one package the V3 notebook does not: **SpikingJelly**.
The simplest route is to extend the `sentrix_v2` environment; a dedicated env is cleaner if
you want to pin the versions the official repository uses.

## Option A — add SpikingJelly to the existing environment

```bat
conda activate sentrix_v2
pip install spikingjelly
python -c "from spikingjelly.activation_based import neuron, functional, surrogate, layer; print('spikingjelly OK')"
```

## Option B — dedicated environment (recommended for reproducing the benchmark)

```bat
conda deactivate
conda create -n sentrix_dvs python=3.11 -y
conda activate sentrix_dvs
python -m pip install --upgrade pip setuptools wheel

:: match the CUDA build to your driver (nvidia-smi, top-right)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

pip install spikingjelly timm
pip install numpy pandas scikit-learn matplotlib seaborn tqdm
pip install jupyterlab notebook ipykernel ipywidgets

python -m ipykernel install --user --name sentrix_dvs --display-name "Python (sentrix_dvs)"

python -c "import torch; from spikingjelly.activation_based import neuron; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"
```

The official repository pins `spikingjelly==0.0.0.0.12`, `torch==1.12.1` and `timm==0.6.12`.
Those pins are only necessary if you intend to run **their** code; this notebook uses the
stable `spikingjelly.activation_based` API and works with current releases.

## Disk and memory notes

- The `.npz` dataset stays on `G:` — nothing is copied. Budget roughly
  `n_videos × 4 MB` for the tensor cache under `G:\Sentrix\training\cache_ucf_crime_dvs`.
- Building the cache is the slow part (one full decompression per video). It is resumable —
  rerun CELL 10 after any interruption and it skips what already exists.
- VRAM scales with `BATCH_VIDEOS × N_SEGMENTS`. At `BATCH_VIDEOS=4`, `N_SEGMENTS=32`,
  `TARGET_HW=128` the encoder sees 256 event frames per step — roughly 8 GB. Halve
  `BATCH_VIDEOS` or `N_SEGMENTS` on a smaller card.
- For a fast end-to-end rehearsal set `MAX_VIDEOS_PER_SPLIT = 40` in CELL 10 and
  `EPOCHS = 3` in CELL 12.